In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from io import StringIO
from tqdm import tqdm

## Helper Classes

In [ ]:
class AnomalyEDA:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_data = None
        sns.set_style('whitegrid')
        plt.rcParams['figure.figsize'] = (12, 6)

    def import_data(self):
        """Load cleaned data from S3."""
        print(f"Loading data from S3: s3://{self.str_bucket}/00_data_collection/paysim_clean.csv")
        str_uri = f's3://{self.str_bucket}/00_data_collection/paysim_clean.csv'
        self.df_data = pd.read_csv(str_uri)
        print(f"Data loaded: {self.df_data.shape}")
        return self.df_data

    def get_df_info(self):
        """Print basic statistics about the dataset."""
        if self.df_data is None:
            raise ValueError("Data not loaded. Call import_data() first.")
        
        print("\n=== DATASET INFORMATION ===")
        print(f"Shape: {self.df_data.shape}")
        print(f"Memory usage: {self.df_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")
        print(f"\nNull values: {self.df_data.isnull().sum().sum()}")
        print(f"\nData types:")
        print(self.df_data.dtypes)
        print(f"\nFraud rate:")
        flt_fraud_rate = self.df_data['isFraud'].mean()
        print(f"  Non-fraud: {(self.df_data['isFraud'] == 0).sum():,} ({(1 - flt_fraud_rate) * 100:.2f}%)")
        print(f"  Fraud: {(self.df_data['isFraud'] == 1).sum():,} ({flt_fraud_rate * 100:.4f}%)")

    def plot_class_distribution(self):
        """Plot fraud vs non-fraud class distribution."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        fig, ax = plt.subplots(figsize=(10, 6))
        dict_counts = self.df_data['isFraud'].value_counts().sort_index()
        dict_labels = {0: 'Non-Fraud', 1: 'Fraud'}
        list_labels = [dict_labels[i] for i in dict_counts.index]
        
        bars = ax.bar(list_labels, dict_counts.values, color=['#2ecc71', '#e74c3c'], alpha=0.7, edgecolor='black')
        ax.set_ylabel('Count', fontsize=12, fontweight='bold')
        ax.set_title('Class Distribution: Fraud vs Non-Fraud Transactions', fontsize=14, fontweight='bold')
        ax.set_yscale('log')
        
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height):,}\n({height/len(self.df_data)*100:.2f}%)',
                    ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_class_distribution.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 01_class_distribution.png")

    def plot_transaction_type_distribution(self):
        """Plot transaction types."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        fig, ax = plt.subplots(figsize=(12, 6))
        dict_type_counts = self.df_data['type'].value_counts()
        
        bars = ax.bar(dict_type_counts.index, dict_type_counts.values, color='steelblue', alpha=0.7, edgecolor='black')
        ax.set_ylabel('Count', fontsize=12, fontweight='bold')
        ax.set_title('Transaction Type Distribution', fontsize=14, fontweight='bold')
        ax.set_xticklabels(dict_type_counts.index, rotation=45, ha='right')
        
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height):,}',
                    ha='center', va='bottom', fontweight='bold', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_transaction_type_distribution.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 02_transaction_type_distribution.png")

    def plot_amount_distribution(self):
        """Plot amount distribution, colored by fraud status."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        ax.hist(self.df_data[self.df_data['isFraud'] == 0]['amount'], bins=100, alpha=0.6, label='Non-Fraud', color='#2ecc71')
        ax.hist(self.df_data[self.df_data['isFraud'] == 1]['amount'], bins=100, alpha=0.6, label='Fraud', color='#e74c3c')
        
        ax.set_xlabel('Amount', fontsize=12, fontweight='bold')
        ax.set_ylabel('Count', fontsize=12, fontweight='bold')
        ax.set_title('Transaction Amount Distribution (Log Scale)', fontsize=14, fontweight='bold')
        ax.set_xscale('log')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_amount_distribution.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 03_amount_distribution.png")

    def plot_amount_by_type(self):
        """Plot box plots of amount by transaction type."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        list_types = sorted(self.df_data['type'].unique())
        list_data = [self.df_data[self.df_data['type'] == t]['amount'].values for t in list_types]
        
        bp = ax.boxplot(list_data, labels=list_types, patch_artist=True)
        
        for patch in bp['boxes']:
            patch.set_facecolor('lightblue')
            patch.set_alpha(0.7)
        
        ax.set_ylabel('Amount (log scale)', fontsize=12, fontweight='bold')
        ax.set_title('Amount Distribution by Transaction Type', fontsize=14, fontweight='bold')
        ax.set_yscale('log')
        ax.set_xticklabels(list_types, rotation=45, ha='right')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_amount_by_type.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 04_amount_by_type.png")

    def plot_fraud_by_type(self):
        """Plot fraud rate by transaction type."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        df_fraud_by_type = self.df_data.groupby('type')['isFraud'].agg(['sum', 'count'])
        df_fraud_by_type['fraud_rate'] = (df_fraud_by_type['sum'] / df_fraud_by_type['count'] * 100).sort_values(ascending=False)
        
        bars = ax.bar(df_fraud_by_type.index, df_fraud_by_type['fraud_rate'].values, color='coral', alpha=0.7, edgecolor='black')
        ax.set_ylabel('Fraud Rate (%)', fontsize=12, fontweight='bold')
        ax.set_title('Fraud Rate by Transaction Type', fontsize=14, fontweight='bold')
        ax.set_xticklabels(df_fraud_by_type.index, rotation=45, ha='right')
        
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}%',
                    ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/05_fraud_by_type.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 05_fraud_by_type.png")

    def plot_balance_analysis(self):
        """Plot balance changes, colored by fraud status."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        df_sample = self.df_data.sample(n=min(10000, len(self.df_data)), random_state=42)
        df_sample['balance_change_orig'] = df_sample['newbalanceOrig'] - df_sample['oldbalanceOrg']
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        scatter_normal = ax.scatter(df_sample[df_sample['isFraud'] == 0]['amount'],
                                    df_sample[df_sample['isFraud'] == 0]['balance_change_orig'],
                                    alpha=0.3, s=10, color='#2ecc71', label='Non-Fraud')
        scatter_fraud = ax.scatter(df_sample[df_sample['isFraud'] == 1]['amount'],
                                   df_sample[df_sample['isFraud'] == 1]['balance_change_orig'],
                                   alpha=0.7, s=30, color='#e74c3c', label='Fraud', edgecolors='darkred')
        
        ax.set_xlabel('Amount', fontsize=12, fontweight='bold')
        ax.set_ylabel('Balance Change (Origin Account)', fontsize=12, fontweight='bold')
        ax.set_title('Balance Changes vs Amount (Sample)', fontsize=14, fontweight='bold')
        ax.set_xscale('log')
        ax.set_yscale('symlog')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/06_balance_analysis.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 06_balance_analysis.png")

    def plot_correlation_heatmap(self):
        """Plot correlation matrix of numeric features."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        df_numeric = self.df_data.select_dtypes(include=[np.number])
        df_corr = df_numeric.corr()
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                    square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
        ax.set_title('Correlation Matrix - Numeric Features', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/07_correlation_heatmap.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 07_correlation_heatmap.png")

    def plot_step_distribution(self):
        """Plot transaction volume over time with fraud overlay."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        df_step_counts = self.df_data.groupby('step')['isFraud'].agg(['count', 'sum'])
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
        
        # Total transactions
        ax1.fill_between(df_step_counts.index, df_step_counts['count'].values, alpha=0.5, color='steelblue')
        ax1.set_ylabel('Transaction Count', fontsize=11, fontweight='bold')
        ax1.set_title('Transaction Volume Over Time (Steps)', fontsize=13, fontweight='bold')
        ax1.grid(True, alpha=0.3)
        
        # Fraud count
        ax2.fill_between(df_step_counts.index, df_step_counts['sum'].values, alpha=0.5, color='#e74c3c')
        ax2.set_xlabel('Step (Time)', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Fraud Count', fontsize=11, fontweight='bold')
        ax2.set_title('Fraud Incidents Over Time', fontsize=13, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/08_step_distribution.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 08_step_distribution.png")

## Constants

In [ ]:
str_bucket = 'anomaly-detection-demo'
str_task = 'eda'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

## Load Data

In [ ]:
eda = AnomalyEDA(str_bucket, str_dirname_output)
df_data = eda.import_data()

## Dataset Information

In [ ]:
eda.get_df_info()

In [ ]:
print("\nFirst 5 rows:")
print(df_data.head())

## Class Distribution Analysis

In [ ]:
eda.plot_class_distribution()

## Transaction Type Analysis

In [ ]:
eda.plot_transaction_type_distribution()

In [ ]:
eda.plot_fraud_by_type()

## Amount Analysis

In [ ]:
eda.plot_amount_distribution()

In [ ]:
eda.plot_amount_by_type()

## Balance Analysis

In [ ]:
eda.plot_balance_analysis()

## Correlation Analysis

In [ ]:
eda.plot_correlation_heatmap()

## Temporal Analysis

In [ ]:
eda.plot_step_distribution()

## Summary

In [ ]:
print("\n=== EDA COMPLETE ===")
print(f"Generated {len([f for f in os.listdir(str_dirname_output) if f.endswith('.png')])} visualizations")
print("All plots saved to ./output/")